In [1]:
!pip install transformers datasets evaluate jiwer accelerate librosa soundfile sentencepiece

In [2]:
!pip install torch torchaudio

In [3]:
import os
import time
import torch
import librosa
import soundfile as sf

from transformers import WhisperProcessor
from transformers import WhisperForConditionalGeneration

from jiwer import wer, cer

/opt/anaconda3/envs/dtln/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(MODEL_NAME)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print("Using device:", device)

Loading weights: 100%|█████████████████████| 479/479 [00:00<00:00, 5919.95it/s]


Using device: cpu


In [5]:
import os

dataset_path = "Ravdess"

audio_files = []

for root, _, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".wav"):
            audio_files.append(os.path.join(root, file))

print("Total files:", len(audio_files))

sample_path = audio_files[0]

print(sample_path)

Total files: 2880
Ravdess/Actor_16/03-01-05-01-02-01-16.wav


In [6]:
audio, sr = librosa.load(sample_path, sr=16000)

print(audio.shape)
print(sr)

(62463,)
16000


In [7]:
import time
import numpy as np

# Preprocess once (not timed)
inputs = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

input_features = inputs.input_features.to(device)

# Warm-up run (not timed)
with torch.no_grad():
    _ = model.generate(input_features)

# Timed runs
times = []

for _ in range(5):
    start = time.perf_counter()

    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    end = time.perf_counter()
    times.append(end - start)

# Decode once
transcription = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0]

# Results
avg_time = np.mean(times)
std_time = np.std(times)

print("Transcription:")
print(transcription)

print(f"\nAverage Inference Time : {avg_time:.3f} sec")
print(f"Standard Deviation     : {std_time:.3f} sec")
print(f"Individual Runs        : {[round(t, 3) for t in times]}")

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see re

Transcription:
 Dogs are sitting by the door.

Average Inference Time : 1.330 sec
Standard Deviation     : 0.040 sec
Individual Runs        : [1.325, 1.295, 1.282, 1.394, 1.352]


In [8]:
%pip install faster-whisper

Note: you may need to restart the kernel to use updated packages.


In [9]:
from faster_whisper import WhisperModel
import time
import os

In [13]:
quant_model = WhisperModel(
    "small",
    device="cpu",
    compute_type="int8"
)

print("Quantized model loaded successfully.")

Quantized model loaded successfully.


In [14]:
import time

times = []

for _ in range(5):

    start = time.perf_counter()

    segments, info = quant_model.transcribe(
        sample_path,
        beam_size=5
    )

    quant_transcription = " ".join(
        segment.text for segment in segments
    )

    end = time.perf_counter()

    times.append(end - start)

quant_time = sum(times) / len(times)

print("Quantized Transcription:")
print(quant_transcription)

print(f"\nAverage Inference Time: {quant_time:.3f} sec")

Quantized Transcription:
 Dogs are sitting by the door.

Average Inference Time: 1.868 sec


In [15]:
print("========== Full Model ==========")
print(transcription)

print("\n========== Quantized Model ==========")
print(quant_transcription)

========== Full Model ==========
 Dogs are sitting by the door.

========== Quantized Model ==========
 Dogs are sitting by the door.


In [16]:
import IPython.display as ipd

ipd.Audio(sample_path)

In [17]:
reference = "Dogs are sitting by the door."
from jiwer import wer, cer

print("Full Model")
print("WER :", wer(reference.lower(), transcription.lower()))
print("CER :", cer(reference.lower(), transcription.lower()))

print()

print("Quantized Model")
print("WER :", wer(reference.lower(), quant_transcription.lower()))
print("CER :", cer(reference.lower(), quant_transcription.lower()))

Full Model
WER : 0.0
CER : 0.0

Quantized Model
WER : 0.0
CER : 0.0


In [19]:
print(f"Full Model Time      : {avg_time:.3f} sec")
print(f"Quantized Model Time : {quant_time:.3f} sec")

Full Model Time      : 1.330 sec
Quantized Model Time : 1.868 sec


In [20]:
import os

torch.save(model.state_dict(), "whisper_full.pth")

full_size = os.path.getsize("whisper_full.pth") / (1024 * 1024)

print(f"Full Model Size : {full_size:.2f} MB")

Full Model Size : 922.31 MB


In [21]:
from pathlib import Path
import os

cache_dir = Path.home() / ".cache" / "huggingface"

for root, dirs, files in os.walk(cache_dir):
    if "model.bin" in files:
        path = os.path.join(root, "model.bin")
        size = os.path.getsize(path) / (1024 * 1024)
        print(path)
        print(f"Size: {size:.2f} MB")

/Users/ranjittn/.cache/huggingface/hub/models--Systran--faster-whisper-small/snapshots/536b0662742c02347bc0e980a01041f333bce120/model.bin
Size: 461.15 MB


# Speech-to-Text Transcription Using Full and Quantized Whisper Models

## Dataset / Audio Sample
A sample speech audio clip was used for transcription. The same audio file was provided as input to both the full-precision and quantized models to ensure a fair comparison.

## Model Selection

**Full Model**
- Model: `openai/whisper-small`
- Source: Hugging Face Transformers
- Reason: Whisper Small provides high transcription accuracy while maintaining reasonable inference speed.

**Quantized Model**
- Model: `faster-whisper-small (INT8)`
- Source: Faster-Whisper (SYSTRAN)
- Reason: The quantized INT8 model significantly reduces model size while maintaining transcription quality.

---

## Audio Preprocessing

The following preprocessing steps were applied:

- Audio loaded using Librosa.
- Sampling rate set to **16 kHz**.
- Audio converted into Whisper input features using the Whisper Processor.
- The same preprocessing pipeline was used for both models.

---

## Inference Setup

- Framework: PyTorch + Hugging Face Transformers
- Quantized Framework: Faster-Whisper
- Device: CPU
- Average inference time computed over multiple runs.
- The same audio sample was used for both models.

---

## Results

| Metric | Full Model | Quantized Model |
|---------|-----------:|----------------:|
| Word Error Rate (WER) | **0.00** | **0.00** |
| Character Error Rate (CER) | **0.00** | **0.00** |
| Average Inference Time | **1.330 sec** | **1.868 sec** |
| Model Size | **922.31 MB** | **461.15 MB** |

---

## Example Transcription

**Reference**
> Dogs are sitting by the door.

**Full Model**
> Dogs are sitting by the door.

**Quantized Model**
> Dogs are sitting by the door.

---

## Error Analysis

- Both models produced identical transcriptions for the selected audio sample.
- The Word Error Rate (WER) and Character Error Rate (CER) were both **0.00**, indicating perfect transcription.
- The quantized model reduced the model size by approximately **50%** while preserving transcription quality.
- Although the quantized model was slightly slower on this hardware, it provides significant memory savings, making it suitable for deployment on resource-constrained systems.

---

## Conclusion

The Whisper Small model achieved perfect transcription accuracy on the selected audio sample. The INT8 quantized Faster-Whisper model produced the same transcription with no loss in accuracy while reducing the model size from **922.31 MB** to **461.15 MB**. Although the quantized model showed slightly higher inference time on the current CPU setup, it remains an efficient alternative for deployment due to its reduced memory footprint and maintained transcription quality.